# 🟡 Interview: ALiBi Attention

Interview solution for ALiBi (Attention with Linear Biases).

$$m_h = \frac{1}{2^{8h/H}}, \quad \text{bias}[h,i,j] = -m_h |i-j|$$

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def alibi_attention(Q, K, V, num_heads):
    """
    手写 ALiBi 注意力 —— 面试版。

    面试考点:
    1. ALiBi 用固定线性偏置替代可学习位置编码，偏置 = -m_h * |i-j|
    2. 斜率公式 m_h = 2^(-8h/H)，几何级数递减
    3. 手写 softmax 是高频考点：exp(x - max) 保证数值稳定
    4. 多头注意力的 reshape + transpose 操作

    参数: Q, K, V: (B, S, D), num_heads: int
    返回: (B, S, D)
    """
    B, S, D = Q.shape
    d_h = D // num_heads

    # ---- Step 1: 斜率 ----
    h_idx = torch.arange(1, num_heads + 1, dtype=torch.float32, device=Q.device)
    slopes = 1.0 / (2.0 ** (8.0 * h_idx / num_heads))  # (H,)

    # ---- Step 2: 距离矩阵 ----
    pos = torch.arange(S, device=Q.device).float()
    dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()  # (S, S)

    # ---- Step 3: 偏置 ----
    bias = -slopes.view(num_heads, 1, 1) * dist.unsqueeze(0)  # (H, S, S)

    # ---- Step 4: 多头拆分 ----
    Qh = Q.view(B, S, num_heads, d_h).transpose(1, 2)  # (B, H, S, d_h)
    Kh = K.view(B, S, num_heads, d_h).transpose(1, 2)
    Vh = V.view(B, S, num_heads, d_h).transpose(1, 2)

    # ---- Step 5: 注意力分数 + 偏置 ----
    scores = (Qh @ Kh.transpose(-2, -1)) / (d_h ** 0.5) + bias.unsqueeze(0)
    # scores: (B, H, S, S)

    # ---- Step 6: 手写 softmax（不用 torch.softmax） ----
    # 数值稳定: 先减去每行最大值，防止 exp 溢出
    # max_vals: (B, H, S, 1)，keepdim=True 保持维度便于广播
    max_vals = scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(scores - max_vals)  # (B, H, S, S)
    sum_exp = exp_scores.sum(dim=-1, keepdim=True)  # (B, H, S, 1)
    attn = exp_scores / sum_exp  # (B, H, S, S)

    # ---- Step 7: 加权求和 ----
    out = (attn @ Vh).transpose(1, 2).reshape(B, S, D)
    return out

In [ ]:
# Verify
torch.manual_seed(0)
B, S, D, H = 2, 6, 16, 4
Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

out = alibi_attention(Q, K, V, num_heads=H)
print("Output shape:", out.shape)

h_idx = torch.arange(1, H + 1, dtype=torch.float32)
slopes = 1.0 / (2.0 ** (8.0 * h_idx / H))
print("Slopes for 4 heads:", slopes)

In [ ]:
from torch_judge import check
check("alibi")

## 📝 核心思路总结

1. **ALiBi 的核心创新**：用固定的线性偏置 `-m_h * |i-j|` 替代可学习的位置编码，直接加到注意力分数上，远距离 token 自然获得更低的注意力权重。
2. **几何级数斜率**：`m_h = 2^(-8h/H)` 使不同头关注不同距离范围——小斜率头看远距离，大斜率头看近距离，实现多尺度注意力。
3. **数值稳定 softmax**：`exp(x - max(x))` 是标准技巧，面试必考。减去最大值不改变 softmax 结果，但防止 exp 溢出。
4. **多头 reshape 技巧**：`(B, S, D) → (B, S, H, d_h) → (B, H, S, d_h)`，先 view 拆维度再 transpose 调整顺序，是 PyTorch 多头注意力的标准写法。